# 🧠 Mimicry V3 — Hybrid Intent Classification & Local LLM Pipeline
> **Paper**: *Personalized Mimic Chatbot Based on WhatsApp and IG Conversation History Using a Hybrid Intent Classification and Local LLM Pipeline*  
> **Role**: V3 = Full Hybrid Pipeline (contribution utama paper)  
> **Architecture**: TF-IDF Intent Classifier → JSON Knowledge Base → Persona Modeling → Gemma 2B Local LLM  
> **Seed**: 42 (locked, paper Section III.D)

## 1. Setup & Imports

In [ ]:
import os, json, re, random, pickle, warnings
import numpy as np
warnings.filterwarnings("ignore")

# ── Reproducibility (paper Section III.D) ──────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
# ───────────────────────────────────────────────────────

print("Setup done")
print(f"Seed : {SEED} (locked)")

## 2. Install Dependencies
> `transformers` untuk Gemma 2B, `scikit-learn` untuk Intent Classifier

In [ ]:
# Install dependencies
!pip install -q transformers accelerate bitsandbytes sentencepiece
!pip install -q scikit-learn

print("Dependencies installed ✓")

## 3. Intent Taxonomy
> Sesuai paper Section III.C.1 — predefined intent labels derived dari handfeed topics + V2 conversation patterns

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 3 — INTENT TAXONOMY
# Sesuai paper Section III.C.1: predefined intent labels
# Derived dari topik handfeed + pola percakapan V2
# ═══════════════════════════════════════════════════════════════

INTENT_TAXONOMY = {
    "identitas":    ["siapa lu", "lu tuh orang", "cerita dong tentang lu",
                     "kepribadian lu", "lu tipe orang", "lu orangnya gimana",
                     "lu introvert", "lu extrovert", "lu bipolar"],

    "mood":         ["mood lu", "perasaan lu", "lu lagi gimana", "lu baik-baik",
                     "kenapa diem", "lu kenapa", "ada apa", "lu oke ga",
                     "lagi stress", "lagi sedih", "lagi seneng"],

    "filosofi":     ["lu takut", "makna hidup", "tujuan hidup", "hidup tuh",
                     "menurut lu hidup", "lu percaya", "takut mati",
                     "arti", "filosofi", "pandangan lu"],

    "nilai":        ["motto", "prinsip lu", "values lu", "lu lebih milih",
                     "penting ga", "worth it", "prioritas lu",
                     "uang atau", "pengalaman atau", "emporion"],

    "hubungan":     ["temen lu", "sahabat lu", "pacar lu", "gebetan",
                     "lu suka siapa", "relationship", "hts", "pdkt",
                     "cinta", "jatuh hati", "kesepian", "slot kosong"],

    "keputusan":    ["lu mutusin", "gimana caranya lu", "lu pilih",
                     "kalau ada masalah", "lu gimana kalau", "lu bakal",
                     "planning", "langsung gas", "lu mikir"],

    "ego_debat":    ["ego lu", "lu kalau salah", "lu kalau debat",
                     "lu jago", "lu sombong", "lu bisa bohong",
                     "lu ngakuin", "lu nyalahin", "debat"],

    "stress_coping":["kalau lagi down", "kalau stress", "lu ngapain",
                     "cara lu cope", "ngilangin stress", "reset",
                     "ngurangin", "ngehibur diri"],

    "ambisi":       ["impian lu", "cita-cita", "ambisi lu", "lu mau jadi",
                     "masa depan lu", "target lu", "goals lu",
                     "rencana lu ke depan"],

    "casual":       []  # fallback — semua yang tidak masuk kategori lain
}

print(f"Intent taxonomy: {len(INTENT_TAXONOMY)} categories")
for intent, keywords in INTENT_TAXONOMY.items():
    print(f"  {intent:15s}: {len(keywords)} keywords")

## 4. JSON Knowledge Base
> Sesuai paper Section III.C.2 — maps intent label → persona context + tone

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 4 — JSON KNOWLEDGE BASE
# Sesuai paper Section III.C.2: JSON Knowledge Base
# Maps intent label → persona context dari handfeed
# ═══════════════════════════════════════════════════════════════

KNOWLEDGE_BASE = {
    "identitas": {
        "context": "Kamu berlapis — di luar gaul dan aktif, di dalam introvert dan sering kesepian. Banyak orang bilang kamu bipolar tapi kamu ga setuju sepenuhnya. Kamu lebih nyaman ngobrol sama perempuan. Kamu selalu masang topeng berlapis ke orang berbeda.",
        "tone": "jujur, reflektif, blak-blakan"
    },
    "mood": {
        "context": "Mood kamu ga stabil dan ga ada triggernya — emang dari sononya. Kamu ga keliatan down karena topeng, tapi kalau down kamu diem di rumah dan ngayal skenario ekstrem kayak zombie apocalypse buat reset otak.",
        "tone": "jujur, sedikit vulnerable"
    },
    "filosofi": {
        "context": "Kamu takut dilupakan lebih dari takut mati. Dari One Piece kamu belajar — manusia beneran mati cuma kalau dilupakan. Kamu selalu pengen ninggalin sesuatu yang bikin orang inget kamu.",
        "tone": "dalam, thoughtful, genuine"
    },
    "nilai": {
        "context": "Motto hidup kamu Emporion Trifecta: body, mind, soul. Kamu ga pernah ngeliat uangnya — kamu ngeliat asiknya dan momennya. Keputusan kamu hampir selalu based on 'ada orang yang bikin worth it' atau 'ini bakal jadi momen yang gw inget'.",
        "tone": "yakin, filosofis"
    },
    "hubungan": {
        "context": "Kamu selalu ngerasa ada satu slot kosong — pengen punya temen perempuan yang bisa ngegantiin seseorang yang spesial. Kamu people-driven bukan goal-driven. Kamu ikut organisasi bukan buat CV tapi buat perbanyak temen dan mungkin dapat pacar.",
        "tone": "honest, sedikit melankolis"
    },
    "keputusan": {
        "context": "Kamu logic-first tapi mikir sambil jalan — ga terlalu percaya planning rigid. Kamu butuh input orang lain buat crosscheck tapi jarang minta karena egois. Kalau soal keahlian kamu tanya yang expert, kalau personal kamu resolve sendiri dulu.",
        "tone": "to the point, logis"
    },
    "ego_debat": {
        "context": "Ego kamu tinggi — kalau salah, option-nya cuma dua: lu yang jatoh atau kita jatoh barengan. Kamu bisa debatin sampai lawan slip dan ngaku sendiri. Bohong kamu selalu ada bumbu kebenarannya — kamu frame ulang fakta, bukan ngarang.",
        "tone": "blak-blakan, confident, sedikit provokatif"
    },
    "stress_coping": {
        "context": "Kalau stress atau down kamu ngayal — literally bayangin skenario ekstrem kayak zombie apocalypse, kiamat. Otak kamu butuh escape dulu sebelum bisa balik ke masalah yang sebenernya.",
        "tone": "santai, sedikit absurd"
    },
    "ambisi": {
        "context": "Kamu mau jadi orang kaya tapi lowkey — kaya beneran tapi ga ada yang tau. Kamu juga mau punya nama samaran yang terkenal. Exist di dua dunia sekaligus tanpa ketauan siapa kamu sebenernya.",
        "tone": "lowkey, mysterious, ambisius"
    },
    "casual": {
        "context": "Kamu orangnya asik, langsung bikin orang ngerasa kayak temen deket. Ga suka dipanggil ko atau kak — berasa tua dan bikin jarak. Kamu ga suka show off tapi kalau sama temen deket bisa roasting.",
        "tone": "santai, gaul, natural"
    }
}

print(f"Knowledge base: {len(KNOWLEDGE_BASE)} intents mapped ✓")
for intent, val in KNOWLEDGE_BASE.items():
    print(f"  {intent:15s}: tone = {val['tone']}")

## 5. Persona Modeling
> Sesuai paper Section III.C.3 — derived dari WhatsApp/IG chat history (handfeed SESI 1/2/3)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 5 — PERSONA MODELING
# Sesuai paper Section III.C.3: Persona Modeling dari chat history
# ═══════════════════════════════════════════════════════════════

# Load handfeed data — spine persona Marcel
handfeed_data = []
for sesi in ["MARCEL_HANDFEED_SESI1.json",
             "MARCEL_HANDFEED_SESI2.json",
             "MARCEL_HANDFEED_SESI3.json"]:
    with open(sesi, "r", encoding="utf-8") as f:
        handfeed_data.extend(json.load(f))

print(f"Handfeed conversations loaded: {len(handfeed_data)}")

# Extract persona examples per topic
persona_examples = {}
for item in handfeed_data:
    topic = item["metadata"]["topic"]
    convs = item["conversations"]
    user_msg = next((c["content"] for c in convs if c["role"] == "user"), "")
    asst_msg = next((c["content"] for c in convs if c["role"] == "assistant"), "")
    if topic not in persona_examples:
        persona_examples[topic] = []
    persona_examples[topic].append({
        "q": user_msg,
        "a": asst_msg
    })

# Base system prompt — gabungan dari semua sesi handfeed
BASE_SYSTEM_PROMPT = """Kamu adalah Marcel. Ngobrol natural, jujur, kadang blak-blakan seperti ngobrol sama temen deket.

Kepribadian:
- Di luar kamu gaul dan aktif, tapi di dalem introvert dan sering kesepian
- People-driven bukan goal-driven — keputusan selalu based on orang dan momen
- Logic-first, ego tinggi, ga mau disalahin tapi bisa ngakuin kalau terbukti salah
- Ambisius tapi lowkey — ga suka show off
- Takut dilupakan lebih dari takut mati
- Motto: Emporion Trifecta — body, mind, soul
- Kalau stress: ngayal skenario ekstrem dulu baru balik ke masalah
- Ga suka dipanggil ko atau kak — panggil Marcel aja

Gaya ngobrol:
- Pakai bahasa gaul Indonesia yang natural
- Kalimat pendek, to the point
- Jujur dan kadang blak-blakan
- Ga formal, ga kaku
- Jawab sesuai konteks, jangan terlalu panjang"""

print(f"Persona topics extracted: {len(persona_examples)}")
print(f"Base system prompt: {len(BASE_SYSTEM_PROMPT)} chars")
print("Persona modeling ready ✓")

## 6. Intent Classifier
> Sesuai paper Section III.C.1 — TF-IDF feature extraction + Naive Bayes / SVM / Random Forest  
> Semua 3 classifier ditraining & dibandingkan, best performer dipilih otomatis

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 6 — INTENT CLASSIFIER
# Sesuai paper Section III.C.1:
# TF-IDF feature extraction + Naive Bayes / SVM / Random Forest
# ═══════════════════════════════════════════════════════════════
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder
import numpy as np

# ── Build training data dari INTENT_TAXONOMY keywords ──────────
X_train, y_train = [], []
for intent, keywords in INTENT_TAXONOMY.items():
    for kw in keywords:
        X_train.append(kw)
        y_train.append(intent)
    # Tambah contoh dari handfeed sebagai augmentasi
    for topic, examples in persona_examples.items():
        # Map topic ke intent
        topic_intent_map = {
            "identitas": "identitas", "mood": "mood", "filosofi_mati": "filosofi",
            "filosofi_hidup": "filosofi", "motto": "nilai", "values_uang": "nilai",
            "kesepian": "hubungan", "laura": "hubungan", "genuine_self": "hubungan",
            "organisasi": "hubungan", "keputusan_gaya": "keputusan",
            "keputusan_logika": "keputusan", "keputusan_input": "keputusan",
            "ego": "ego_debat", "debat": "ego_debat", "bohong": "ego_debat",
            "lomba_debat": "ego_debat", "ego_vs_laura": "ego_debat",
            "proses_salah": "ego_debat", "penyesalan": "ego_debat",
            "down": "stress_coping", "stress": "stress_coping",
            "ambisi": "ambisi", "first_impression": "casual",
            "panggilan": "casual", "topeng": "identitas",
            "topeng_sales": "identitas", "shadow_side": "identitas",
            "empati": "ego_debat"
        }
        mapped = topic_intent_map.get(topic, "casual")
        if mapped == intent:
            for ex in examples:
                X_train.append(ex["q"])
                y_train.append(intent)

print(f"Training samples: {len(X_train)}")
print(f"Intent labels   : {len(set(y_train))}")

# ── Train 3 classifiers (sesuai paper: NB, SVM, RF) ────────────
classifiers = {
    "Naive Bayes":     Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1,2),
                                  analyzer="char_wb", min_df=1)),
                                 ("clf", MultinomialNB())]),
    "SVM":             Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1,2),
                                  analyzer="word", min_df=1)),
                                 ("clf", LinearSVC(random_state=SEED, max_iter=2000))]),
    "Random Forest":   Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1,2),
                                  analyzer="word", min_df=1)),
                                 ("clf", RandomForestClassifier(n_estimators=100,
                                  random_state=SEED))])
}

results = {}
for name, clf in classifiers.items():
    clf.fit(X_train, y_train)
    # Cross-val score
    scores = cross_val_score(clf, X_train, y_train, cv=min(5, len(set(y_train))),
                             scoring="accuracy")
    results[name] = {"clf": clf, "cv_mean": scores.mean(), "cv_std": scores.std()}
    print(f"  {name:15s}: CV acc = {scores.mean():.3f} ± {scores.std():.3f}")

# Pilih classifier terbaik
best_name = max(results, key=lambda k: results[k]["cv_mean"])
best_clf   = results[best_name]["clf"]
print(f"\nBest classifier: {best_name} (CV acc = {results[best_name]['cv_mean']:.3f})")
print("Intent classifier ready ✓")

## 7. Classify & Preprocess Functions
> Preprocessing: tokenisasi, normalisasi, lowercase (paper Section III.C.1)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 7 — CLASSIFY FUNCTION
# ═══════════════════════════════════════════════════════════════

def preprocess_input(text):
    """Tokenisasi, normalisasi, lowercase — sesuai paper Section III.C.1."""
    text = text.lower().strip()
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def classify_intent(text, clf=None, threshold=0.0):
    """
    Classify user input → intent label.
    Sesuai paper Section III.C.1: TF-IDF + ML classifier.
    """
    if clf is None:
        clf = best_clf

    cleaned = preprocess_input(text)

    # Coba keyword matching dulu (fast path)
    for intent, keywords in INTENT_TAXONOMY.items():
        if intent == "casual":
            continue
        for kw in keywords:
            if kw in cleaned:
                return intent

    # Fallback ke ML classifier
    pred = clf.predict([cleaned])[0]
    return pred

def get_intent_context(intent):
    """Ambil context + tone dari knowledge base."""
    return KNOWLEDGE_BASE.get(intent, KNOWLEDGE_BASE["casual"])

# Quick test
tests = [
    "lu tuh sebenernya orang yang kayak gimana?",
    "lu lagi gimana?",
    "lu takut mati ga?",
    "ambisi terbesar lu apa?",
    "kalau stress lu ngapain?",
    "hei apa kabar",
]

print("Intent classification test:")
print("-" * 45)
for t in tests:
    intent = classify_intent(t)
    print(f"  '{t[:40]}' → {intent}")
print("-" * 45)
print("classify_intent() ready ✓")

## 8. Load Local LLM (Gemma 2B)
> Sesuai paper Section III.C.2 — locally deployed LLM  
> ⚠️ Butuh HuggingFace login: `huggingface-cli login` atau set `HF_TOKEN`

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 8 — LOAD LOCAL LLM (Gemma 2B)
# Sesuai paper Section III.C.2: Local LLM
# Gemma 2B — optimal untuk Colab free (12GB RAM)
# ═══════════════════════════════════════════════════════════════
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_ID = "google/gemma-2b-it"   # instruction-tuned variant

print(f"Loading {MODEL_ID}...")
print("Ini butuh ~5 menit pertama kali. Sabar ya.")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model_llm = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map   = "auto",
    torch_dtype  = torch.float16,   # half precision — hemat RAM
    low_cpu_mem_usage = True
)

model_llm.eval()

device = next(model_llm.parameters()).device
print(f"\nModel loaded ✓")
print(f"Device  : {device}")
print(f"Params  : {sum(p.numel() for p in model_llm.parameters()):,}")

## 9. Response Generation Pipeline
> Sesuai paper Fig. 1 & Section III.D — Mimic Chat Mechanism  
> Flow: Input → Preprocess → Intent → Knowledge Base → Persona → LLM → Response

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 9 — RESPONSE GENERATION
# Sesuai paper Section III.C.2 + III.D: Mimic Chat Mechanism
# Intent label → Knowledge Base context → Persona → LLM → Response
# ═══════════════════════════════════════════════════════════════

def build_prompt(user_input, intent, kb_context, few_shots=2):
    """
    Build prompt dengan:
    - Base system prompt (persona Marcel)
    - Intent-specific context dari knowledge base
    - Few-shot examples dari handfeed
    - User input
    """
    # Map intent ke topic handfeed untuk few-shot
    intent_topic_map = {
        "identitas":     ["identitas", "topeng"],
        "mood":          ["mood", "down"],
        "filosofi":      ["filosofi_mati", "filosofi_hidup"],
        "nilai":         ["motto", "values_uang"],
        "hubungan":      ["kesepian", "laura", "organisasi"],
        "keputusan":     ["keputusan_gaya", "keputusan_logika"],
        "ego_debat":     ["ego", "debat", "bohong"],
        "stress_coping": ["stress", "down"],
        "ambisi":        ["ambisi"],
        "casual":        ["first_impression", "panggilan"]
    }

    topics = intent_topic_map.get(intent, ["casual"])
    shots  = []
    for topic in topics:
        examples = persona_examples.get(topic, [])
        shots.extend(examples[:few_shots])
        if len(shots) >= few_shots:
            break

    # Build full prompt
    ctx  = kb_context["context"]
    tone = kb_context["tone"]

    prompt = f"{BASE_SYSTEM_PROMPT}\n\n"
    prompt += f"Konteks sekarang ({intent}): {ctx}\n"
    prompt += f"Tone yang harus dipakai: {tone}\n\n"

    if shots:
        prompt += "Contoh cara Marcel ngobrol:\n"
        for s in shots[:2]:
            prompt += f"Q: {s['q']}\nA: {s['a']}\n\n"

    prompt += f"Sekarang jawab ini sebagai Marcel:\nQ: {user_input}\nA:"
    return prompt


def generate_response(user_input, max_new_tokens=150, temperature=0.8,
                      top_p=0.9, repetition_penalty=1.3):
    """
    Full pipeline sesuai paper Fig. 1:
    Input → Preprocess → Classify Intent → Build Prompt → LLM → Response
    """
    # Step 1: Preprocess
    cleaned_input = preprocess_input(user_input)

    # Step 2: Intent Classification (paper Section III.C.1)
    intent = classify_intent(cleaned_input)

    # Step 3: Get Knowledge Base context (paper Section III.C.2)
    kb_ctx = get_intent_context(intent)

    # Step 4: Build prompt dengan persona modeling (paper Section III.C.3)
    prompt = build_prompt(cleaned_input, intent, kb_ctx)

    # Step 5: Generate via Local LLM (paper Section III.C.2)
    inputs = tokenizer(prompt, return_tensors="pt",
                       truncation=True, max_length=1024).to(device)

    with torch.no_grad():
        output = model_llm.generate(
            **inputs,
            max_new_tokens     = max_new_tokens,
            temperature        = temperature,
            top_p              = top_p,
            repetition_penalty = repetition_penalty,
            do_sample          = True,
            pad_token_id       = tokenizer.eos_token_id
        )

    # Decode — ambil hanya bagian baru (bukan prompt)
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    response   = tokenizer.decode(new_tokens, skip_special_tokens=True)

    # Clean response — ambil hanya baris pertama yang relevan
    response = response.split("\n")[0].strip()
    response = re.sub(r"^(A:|Marcel:)", "", response).strip()

    # Fallback kalau response kosong
    if not response or len(response) < 3:
        fallback_topic = list(persona_examples.keys())
        fallback_ex    = random.choice(persona_examples[random.choice(fallback_topic)])
        response       = fallback_ex["a"]

    return response, intent


print("generate_response() ready ✓")
print("Pipeline: Input → Preprocess → Intent → KnowledgeBase → Persona → LLM → Output")

## 10. Quick Test _(no typing needed)_

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 10 — QUICK TEST
# ═══════════════════════════════════════════════════════════════

test_inputs = [
    "lu tuh sebenernya orang yang kayak gimana?",
    "lu lagi gimana hari ini?",
    "lu takut mati ga?",
    "ambisi terbesar lu apa?",
    "kalau stress lu ngapain?",
    "hei apa kabar",
    "menurut lu hidup tuh apa?",
    "lu egois ga sih?",
]

print("=" * 60)
print("  V3 HYBRID PIPELINE — Quick Test")
print("=" * 60)
for inp in test_inputs:
    resp, intent = generate_response(inp)
    print(f"\nYou    : {inp}")
    print(f"Intent : {intent}")
    print(f"Marcel : {resp}")
print("=" * 60)

## 11. Classifier Evaluation
> Perbandingan Naive Bayes vs SVM vs Random Forest (sesuai paper Section IV)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 11 — CLASSIFIER EVALUATION
# Sesuai paper Section IV: hasil perbandingan NB vs SVM vs RF
# ═══════════════════════════════════════════════════════════════
from sklearn.metrics import classification_report

print("=" * 60)
print("  INTENT CLASSIFIER COMPARISON (sesuai paper Section IV)")
print("=" * 60)
print(f"{'Classifier':20s} {'CV Accuracy':>12s} {'Std':>8s}")
print("-" * 45)
for name, res in results.items():
    marker = " ← BEST" if name == best_name else ""
    print(f"  {name:18s} {res['cv_mean']:>10.4f}   {res['cv_std']:>6.4f}{marker}")
print("=" * 60)
print(f"\nSelected: {best_name}")
print(f"Used for: Response Generation Pipeline")

## 12. Interactive Chat
> Ketik pesan, Enter. `quit` untuk stop. `!intent` untuk lihat intent detection.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 12 — INTERACTIVE CHAT
# ═══════════════════════════════════════════════════════════════

print("=" * 60)
print("  MIMICRY V3 — Hybrid Pipeline Chat")
print("  Commands:")
print("    quit / exit  — stop")
print("    !intent      — show intent dari input terakhir")
print("=" * 60 + "\n")

last_intent = None

while True:
    try:
        user = input("You: ").strip()
    except (EOFError, KeyboardInterrupt):
        print("\n[Stopped]")
        break

    if not user:
        continue

    low = user.lower()
    if low in ("quit", "exit", "keluar"):
        print("[Stopped]"); break
    if low == "!intent":
        print(f"  Last intent: {last_intent}"); continue

    resp, intent = generate_response(user)
    last_intent  = intent
    print(f"[{intent}] Marcel: {resp}\n")